# Plot full recording wavelet with episodes

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

In [ ]:
paths = {
    'raw_data': {
        'path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\',
        'extension': 'mat'
    },
    'results': {
        'path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeFeatures',
        'extension': 'csv',
        'suffix': ''
    },
    'output': {
        'path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\FullRecordingWavelet'
    }
}
file_list_table_path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\data_invivo.xlsx'

In [ ]:
# cut_interval = [1650, 1660]  # (50, 250)  # in seconds, or None
cut_interval = None
signal_params = {
    'name': 'wideband',
    'sampling_rate': 1000
}
wavelet_params = {
    'low': 1.0,
    'high': 30.0,
    'clim': [0, 1000],
    'step': 1.0,
    'n_cycles': 7,
    'length': 1.0,
    'addtaper': False,
    'taperlength': 0.1
}
signal_axis_ylim = [-1, 1]
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']
spike_colors = {
    'AIS-AP': 'cell_type',
    'UNKNOWN': '#adadad',
    'RA-AP': '#00ff00'
}
episode_names = ['awake', 'theta', 'sw']
episode_colors = {
    'awake': '#0000ff',
    'theta': '#ff0000',
    'sw': '#00ff00',
    'none': '#adadad'
}
cell_palette = {
    'PYR_L2-3': '#803300ff',
    'PYR_L4-6': '#502d16ff',
    'RSI_L2-3': '#ff48a3ff',
    'RSI_L4-6': '#800080ff',
    'FSI_L2-3': '#d54500ff',
    'FSI_L4-6': '#7a0000ff'
}
show_amplitude = True

Set up result directories.

In [ ]:
from pathlib import Path

results_path = Path(paths['output']['path']) / 'Results'
results_path.mkdir(parents=True, exist_ok=True)

figures_path = Path(paths['output']['path']) / 'Figures'
figures_path.mkdir(parents=True, exist_ok=True)

Load input file list

In [ ]:
input_config = {}
input_config['table_path'] = file_list_table_path
input_config['select_rules'] = [{'wavelet': 1}]

In [ ]:
from workflows.run_util.io import get_cell_info_list
file_list = get_cell_info_list(input_config)
print(f'Found {len(file_list)} cells matching selection criteria.')

In [ ]:
from workflows.run_util.io import collect_path_for_infos
input_files = collect_path_for_infos(file_list, paths)

In [ ]:
def calculate_spike_amplitude_raap_ocurrence(
    spikes: pd.DataFrame,
    time_range: tuple[float, float],
    bin_size: float,
) -> pd.DataFrame:
    """Calculates the average spike amplitude and Ra-Ap ocurrence in a sliding window"""
    edges = np.arange(time_range[0], time_range[1] + bin_size, bin_size)
    amplitudes = spikes['amplitude'].to_numpy()
    raap_counts, _ = np.histogram(
        spikes.loc[spikes['spike_type'] == 'RA-AP', 'timestamp'],
        bins=edges
    )
    raap_freq = raap_counts / bin_size
    spike_bin_indices = np.digitize(spikes['timestamp'], edges) - 1
    spike_amplitudes = np.full_like(raap_counts, np.nan, dtype=np.float64)
    for bin_idx in np.unique(spike_bin_indices):
        if bin_idx < 0 or bin_idx >= len(raap_counts):
            continue
        bin_spikes = amplitudes[spike_bin_indices == bin_idx]
        if bin_spikes.shape[0] > 0:
            spike_amplitudes[bin_idx] = np.mean(bin_spikes)
    return pd.DataFrame({
        'times': np.mean(np.vstack([edges[:-1], edges[1:]]), axis=0),
        'raap_freq': raap_freq,
        'amplitude': spike_amplitudes
    })

In [ ]:


from notebook_utils.recording_wavelet import calculate_spectrogram, get_signal, plot_spectrogram
from signalizer.model.experiment.recordings import MatRecording

n_spike_type = len(spike_types)
bin_size = 1.0  # in seconds

for input_file in tqdm(
    input_files,
    total=len(input_files),
    desc='Processing cells'
):

    recording = MatRecording(input_file.data.raw)
    signal = get_signal(recording, signal_params)
    spectrogram = calculate_spectrogram(signal, wavelet_params)
    spikes = pd.read_csv(input_file.data.processed)

    preprocessed_path = input_file.data.processed
    if show_amplitude:
        fig, axes = plt.subplots(
            4 + len(spike_types) + 2, 1, figsize=(15, 8),
            sharex=True, layout='constrained'
        )
    else:
        fig, axes = plt.subplots(
            4 + len(spike_types), 1, figsize=(15, 8),
            sharex=True, layout='constrained'
        )

    for episode_name in episode_names:
        try:
            episodes = recording.get_episode(episode_name)
        except KeyError:
            continue
        for episode in episodes:
            interval = episode.interval
            axes[0].fill_betweenx(
                (-1, 1), interval.start, interval.end,
                color=episode_colors[episode_name], alpha=0.5
            )

    if cut_interval is not None:
        xlimit = cut_interval
    else:
        xlimit = (signal.times[0], signal.times[-1])
    for ax in axes:
        ax.set_xlim(xlimit)

    plot_spectrogram(
        spectrogram, wavelet_params['clim'], fig, axes[1],
    )

    raw_signal = recording.get_waveform(signal_params['name'])
    axes[2].plot(
        raw_signal.times, raw_signal.values, color='black', linewidth=0.5
    )
    axes[2].set_ylabel('Signal (mV)')
    axes[2].set_ylim(signal_axis_ylim)

    for i, spike_type in enumerate(spike_types):
        subset = spikes[spikes['spike_type'] == spike_type]
        if subset.empty:
            continue

        spike_color = spike_colors[spike_type]
        if spike_color == 'cell_type':
            color = cell_palette[input_file.info.cell_type]
        else:
            color = spike_color

        y_values = np.full(len(subset), i)
        timestamps = subset['timestamp'].to_numpy()
        axes[3].vlines(
            timestamps,
            y_values - 0.45,
            y_values + 0.45,
            color=color,
            alpha=1.0
        )

        axes[4 + n_spike_type - i - 1].plot(
            timestamps[1:], 1 / np.diff(timestamps), color=color, alpha=1.0
        )
        axes[4 + n_spike_type - i - 1].set_ylabel('Inst. Freq.\n(Hz)')
        if show_amplitude:
            amplitude_values = subset['amplitude'].to_numpy()
            axes[-2].plot(
                timestamps,
                amplitude_values,
                '-',
                color=color
            )
            axes[-2].set_ylabel('Amplitude\n(mV)')

    if show_amplitude:
        amplitude_firing = calculate_spike_amplitude_raap_ocurrence(
            spikes,
            (signal.times[0], signal.times[-1]),
            bin_size=bin_size
        )
        non_nan_flag = ~np.isnan(amplitude_firing['raap_freq']) & ~np.isnan(amplitude_firing['amplitude'])
        corr, p_value = pearsonr(amplitude_firing['raap_freq'][non_nan_flag], amplitude_firing['amplitude'][non_nan_flag])
        sp_corr, sp_p_value = spearmanr(amplitude_firing['raap_freq'][non_nan_flag], amplitude_firing['amplitude'][non_nan_flag])
        axes[-1].plot(
            amplitude_firing['times'],
            amplitude_firing['raap_freq'],
            '-',
            color=spike_colors['RA-AP']
        )
        axes[-1].set_ylabel('Ra-AP Freq.\n(Hz)')
        axes[-1].twinx().plot(
            amplitude_firing['times'],
            amplitude_firing['amplitude'],
            '-',
            color='gray'
        )
        axes[-1].text(
            0.02, 0.9,
            f'Pearson r={corr:.2f}, p={p_value:.3f}\nSpearman r={sp_corr:.2f}, p={sp_p_value:.3f}',
            transform=axes[-1].transAxes,
            fontsize=10,
            verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
        )
        axes[-1].twinx().set_ylabel('Spike Amplitude\n(mV)')

    axes[3].set_yticks(range(len(spike_types)))
    axes[3].set_yticklabels(spike_types)
    axes[3].set_ylim((-1, len(spike_types)))

    save_path = figures_path / input_file.info.cell_type
    save_path.mkdir(parents=True, exist_ok=True)
    with plt.rc_context({'svg.fonttype': 'none'}):
        fig.savefig(
            save_path /
            f"{input_file.info.name}_sleep_wavelet.svg",
            format='svg', dpi=100
        )

